In [ ]:
import os
import ast
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
import xgboost as xgb
from datetime import datetime as dt

# 1. Load PCA-Transformed Dataset
# Adjust file path to point directly to your PCA data file
pca_data_path = '../dataset/pca_transformed_data_0.95_target_variance.csv' 
df = pd.read_csv(pca_data_path)

# Separate features, target, and subject groups
X = df.drop(columns=['Activity', 'subject'])
y = df['Activity']
groups = df['subject']

# Integer-encode target labels for XGBoost compatibility
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 2. Keep 20% of subjects as pure hold-out test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=69)
train_idx, test_idx = next(gss.split(X, y_encoded, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

# 3. Parameter Distributions Adjusted for PCA Features
# Reduced feature subsampling ranges since PCA components are orthogonal and ordered by variance
param_distributions = {
    'n_estimators': randint(150, 500),
    'max_depth': randint(3, 9),
    'learning_rate': uniform(0.01, 0.15),
    'subsample': uniform(0.6, 0.4),              # Row sampling (60% to 100%)
    'colsample_bytree': uniform(0.6, 0.4),       # Keep higher fraction of PCA components (60% to 100%)
    'colsample_bylevel': uniform(0.6, 0.4),      # Keep higher fraction per level (60% to 100%)
    'min_child_weight': randint(1, 10),
    'gamma': uniform(0, 1.0),
    'reg_alpha': uniform(0, 5.0),               # L1 Regularization
    'reg_lambda': uniform(1.0, 10.0)            # L2 Regularization
}

# 4. Base Estimator Setup
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=69,
    n_jobs=1
)

# 5. Group Cross-Validation
gkf = GroupKFold(n_splits=5)

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=80,
    scoring='f1_macro',
    cv=gkf,
    verbose=2,
    random_state=69,
    n_jobs=8
)

print(f"Starting search across {X_train.shape[1]} PCA components...")
search.fit(X_train, y_train, groups=groups_train)

# 6. Save All Parameter Runs & Performance Metrics
now_str = dt.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = '../dataset/results'
os.makedirs(output_dir, exist_ok=True)

# A. Save full search history (all 80 hyperparameter configurations with fold/mean metrics)
cv_results_df = pd.DataFrame(search.cv_results_)

# Sort results by best mean validation score for easy evaluation
cv_results_df = cv_results_df.sort_values(by='rank_test_score').reset_index(drop=True)
cv_results_path = os.path.join(output_dir, f'pca_search_all_trials_{now_str}.csv')
cv_results_df.to_csv(cv_results_path, index=False)

# B. Evaluate best estimator on hold-out test set
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

test_macro_f1 = f1_score(y_test, y_pred, average='macro')
test_weighted_f1 = f1_score(y_test, y_pred, average='weighted')

# Save best overall configuration summary
best_summary_df = pd.DataFrame([{
    'timestamp': now_str,
    'best_cv_macro_f1': search.best_score_,
    'test_macro_f1': test_macro_f1,
    'test_weighted_f1': test_weighted_f1,
    'best_params': search.best_params_
}])
best_summary_path = os.path.join(output_dir, f'pca_best_params_summary_{now_str}.csv')
best_summary_df.to_csv(best_summary_path, index=False)

# C. Save hold-out subject predictions
results_df = pd.DataFrame({
    'Subject': groups_test.values,
    'Actual_Activity': le.inverse_transform(y_test),
    'Predicted_Activity': le.inverse_transform(y_pred)
})
predictions_path = os.path.join(output_dir, f'pca_holdout_predictions_{now_str}_xgboost.csv')
results_df.to_csv(predictions_path, index=False)

print("\n" + "="*50)
print("SUCCESS! Tuning complete and results saved.")
print(f"1. All Trial Results: {cv_results_path}")
print(f"2. Best Model Summary: {best_summary_path}")
print(f"3. Test Predictions:   {predictions_path}")
print("="*50)
print(f"Best CV Group Macro-F1:  {search.best_score_:.4f}")
print(f"Hold-out Test Macro-F1:  {test_macro_f1:.4f}")
print("="*50)